# Gold Layer & AI Governance Report

This notebook demonstrates:
- Building business-ready aggregations in the Gold layer
- Revenue analysis by day and category
- Customer lifetime value segmentation
- Running the Claude AI governance agent
- Viewing the generated governance report


In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.utils.spark_session import get_spark_session
from src.utils.paths import GOLD_PATH, REPORTS_PATH

spark = get_spark_session()
sns.set_theme(style='whitegrid')

In [ ]:
# ── Build Gold layer ──────────────────────────────────────────────────────────
from src.pipelines.gold import main as run_gold
run_gold()

In [ ]:
# ── Daily Revenue Trend ───────────────────────────────────────────────────────
daily_pdf = spark.read.format('delta').load(str(GOLD_PATH / 'daily_revenue')).toPandas()
daily_pdf['order_date'] = pd.to_datetime(daily_pdf['order_date'])

daily_total = daily_pdf.groupby('order_date')['gross_revenue'].sum().reset_index()

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(daily_total['order_date'], daily_total['gross_revenue'], linewidth=1.5, color='steelblue')
ax.fill_between(daily_total['order_date'], daily_total['gross_revenue'], alpha=0.15, color='steelblue')
ax.set_title('Daily Gross Revenue (2023-2024)')
ax.set_xlabel('Date')
ax.set_ylabel('Revenue ($)')
plt.tight_layout()
plt.savefig('../governance/reports/daily_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Total revenue: ${daily_pdf["gross_revenue"].sum():,.2f}')
print(f'Total orders: {int(daily_pdf["order_count"].sum()):,}')

In [ ]:
# ── Revenue by Category ───────────────────────────────────────────────────────
cat_rev = daily_pdf.groupby('category')['category_revenue'].sum().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(cat_rev.index, cat_rev.values, color=sns.color_palette('husl', len(cat_rev)))
ax.set_xlabel('Total Revenue ($)')
ax.set_title('Revenue by Product Category')
plt.tight_layout()
plt.show()

In [ ]:
# ── Customer LTV Distribution ────────────────────────────────────────────────
ltv_pdf = spark.read.format('delta').load(str(GOLD_PATH / 'customer_ltv')).toPandas()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LTV distribution
axes[0].hist(ltv_pdf['lifetime_value'].clip(upper=5000), bins=50, color='teal', alpha=0.7)
axes[0].set_xlabel('Lifetime Value ($)')
axes[0].set_ylabel('Customers')
axes[0].set_title('Customer LTV Distribution')

# Tier breakdown
tier_counts = ltv_pdf['ltv_tier'].value_counts()
axes[1].pie(tier_counts.values, labels=tier_counts.index, autopct='%1.1f%%',
            colors=sns.color_palette('husl', 4))
axes[1].set_title('Customer Tier Distribution')

plt.tight_layout()
plt.show()

print(ltv_pdf.groupby('ltv_tier')['lifetime_value'].agg(['count', 'mean', 'sum']).round(2))

In [ ]:
# ── Run AI Governance Agent ───────────────────────────────────────────────────
# Requires ANTHROPIC_API_KEY in .env
from src.agents.governance_agent import main as run_agent
run_agent()

In [ ]:
# ── Display the generated governance report ──────────────────────────────────
from IPython.display import Markdown, display
from pathlib import Path

reports = sorted(REPORTS_PATH.glob('governance_report_*.md'))
if reports:
    report_text = reports[-1].read_text()
    display(Markdown(report_text))
else:
    print('No governance report found. Run the agent first.')